# YData-profiling Example: Baltimore Housing Data

This notebook shows a complete example of using YData-profiling
on a real dataset.

We will:
- load the Baltimore housing dataset
- generate a profiling report
- inspect the dataset
- prepare features for modeling
- train a regression model
- evaluate the model

In [9]:
import sys
from pathlib import Path

sys.path.append(str(Path("src").resolve()))
import ydata_profiling_utils as ydputi

## 1. Load the dataset

We load the Baltimore housing dataset and inspect its basic structure.

In [10]:
df = ydputi.load_baltimore_data()
ydputi.print_basic_info(df)

Data shape: (211, 17)

Columns:
['STATION', 'PRICE', 'NROOM', 'DWELL', 'NBATH', 'PATIO', 'FIREPL', 'AC', 'BMENT', 'NSTOR', 'GAR', 'AGE', 'CITCOU', 'LOTSZ', 'SQFT', 'X', 'Y']

First 5 rows:
   STATION  PRICE  NROOM  DWELL  NBATH  PATIO  FIREPL   AC  BMENT  NSTOR  GAR  \
0        1   47.0    4.0    0.0    1.0    0.0     0.0  0.0    2.0    3.0  0.0   
1        2  113.0    7.0    1.0    2.5    1.0     1.0  1.0    2.0    2.0  2.0   
2        3  165.0    7.0    1.0    2.5    1.0     1.0  0.0    3.0    2.0  2.0   
3        4  104.3    7.0    1.0    2.5    1.0     1.0  1.0    2.0    2.0  2.0   
4        5   62.5    7.0    1.0    1.5    1.0     1.0  0.0    2.0    2.0  0.0   

     AGE  CITCOU   LOTSZ   SQFT      X      Y  
0  148.0     0.0    5.70  11.25  907.0  534.0  
1    9.0     1.0  279.51  28.92  922.0  574.0  
2   23.0     1.0   70.64  30.62  920.0  581.0  
3    5.0     1.0  174.63  26.12  923.0  578.0  
4   19.0     1.0  107.80  22.04  918.0  574.0  


## 2. Generate a profiling report

We generate a full YData-profiling report for the real dataset.
This helps us inspect variable types, distributions, and potential data issues.

In [11]:
profile = ydputi.create_profile_report(
    df,
    title="Baltimore Housing Data Profiling Report"
)
output_path = ydputi.save_profile_report(
    profile,
    output_filename="baltim_example_profile.html"
)
print(f"Report saved to: {output_path}")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 17/17 [00:00<00:00, 138992.53it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Report saved to: /home/songshen/projects/gpsaggese.github.io/class_project/data605/Spring2026/projects/UmdTask391_DATA605_Spring2026_YData_profiling/outputs/baltim_example_profile.html


## 3. Review profiling insights

The generated YData-profiling report helps identify important data quality and modeling issues before building a regression model.

In this dataset, the report is useful for checking:

- variable types and numeric ranges
- missing values
- duplicate rows
- skewed distributions
- correlations between predictors and `PRICE`

These checks help us decide how to clean the data and prepare it for modeling.


## 4. Clean and prepare the data

We clean the dataset by removing duplicate rows and keeping numeric columns for regression modeling.

In [12]:
df_clean = ydputi.clean_baltimore_data(df)

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
df_clean.head()

Original shape: (211, 17)
Cleaned shape: (211, 17)


,STATION,PRICE,NROOM,DWELL,NBATH,PATIO,FIREPL,AC,BMENT,NSTOR,GAR,AGE,CITCOU,LOTSZ,SQFT,X,Y
0,1,47.0,4.0,0.0,1.0,0.0,0.0,0.0,2.0,3.0,0.0,148.0,0.0,5.70,11.25,907.0,534.0
1,2,113.0,7.0,1.0,2.5,1.0,1.0,1.0,2.0,2.0,2.0,9.0,1.0,279.51,28.92,922.0,574.0
2,3,165.0,7.0,1.0,2.5,1.0,1.0,0.0,3.0,2.0,2.0,23.0,1.0,70.64,30.62,920.0,581.0
3,4,104.3,7.0,1.0,2.5,1.0,1.0,1.0,2.0,2.0,2.0,5.0,1.0,174.63,26.12,923.0,578.0
4,5,62.5,7.0,1.0,1.5,1.0,1.0,0.0,2.0,2.0,0.0,19.0,1.0,107.80,22.04,918.0,574.0


## 5. Prepare regression features

The target variable is `PRICE`. All other numeric columns are used as predictors.

In [13]:
X, y = ydputi.prepare_regression_data(
    df_clean,
    target_col="PRICE",
)

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
print("Target variable:", y.name)

Feature matrix shape: (211, 16)
Target vector shape: (211,)
Target variable: PRICE


## 6. Train a regression model

We train a Random Forest regression model. Missing feature values are filled with median values before training.

In [14]:
model, X_train, X_test, y_train, y_test = ydputi.train_regression_model(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 168
Testing rows: 43


## 7. Evaluate the model

We evaluate the model using RMSE and R-squared.

- RMSE measures the typical prediction error in the same unit as `PRICE`.
- R-squared measures how much variation in `PRICE` is explained by the model.

In [15]:
metrics = ydputi.evaluate_regression_model(
    model,
    X_test,
    y_test,
)

for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

RMSE: 12.6307
R2: 0.6915


## 8. Summary

YData-profiling supports the modeling workflow by providing a fast overview of the dataset before model training. The profile report helps identify variable types, missing values, distributions, and correlations. These insights make the cleaning and feature preparation steps more systematic.

In this example, we used the Baltimore housing dataset, generated an automated profile report, cleaned the data, trained a regression model, and evaluated its predictive performance using RMSE and R-squared.